# Computational Linguistics: practical session 2 (21 October 2026) **solutions**

Note: you are not required to submit any assignments for this practical session.

For these exercises, we will make use of the interactive Python environment provided by Google Colab. The environment makes it possible to run Python code within your web browser. It will be helpful if you already know some basic Python (as provided in the first lectures of the course 'Scripting Languages', for example). But even you don't know anything about programming, you should be able to follow along with the examples.

You are highly recommended to use Chrome as a browser to run this notebook. In other browers (firefox, safari, ..), you are likely to run into problems with certain parts (such as the visualization interface below).

## Word embeddings

In this practical session, we'll automatically induce word embeddings based on an extract from Wikipedia. To do so, we'll make use of the Python library called `gensim`. `gensim` is a vector space modeling and topic modeling
toolkit for Python, and it contains an efficient implementation of
the word2vec algorithms for word embedding induction.

Word2vec consists of two different algorithms: skipgram (sg) and
continuous-bag-of-words (cbow). The underlying prediction task of the
former is to estimate the context words from the target word; the
prediction task of the latter is to estimate the target word from the
sum of the context words. More information can be found in the course
slides and in the background material of the course.

First, we'll make sure the `gensim` library is properly installed.

In [ ]:
%pip install gensim

**The shared data cell.** In the second half of this session, we will return to the experiment we started in practical session 1: the same movie-review dataset, with exactly the same split into a *training* set (18,000 reviews), a *development* set (2,000 reviews) and a *test* set (5,000 reviews). The cell below is identical in every notebook: it downloads the data if necessary and rebuilds the split with a fixed random seed, so that the numbers you obtain in different sessions are comparable (and so that you can continue if you missed a session). It also defines a small helper function, `evaluate`, that computes accuracy and macro-averaged F1. Run it now; we will need it later on.

In [ ]:
# ==========================================================================
# SHARED DATA CELL -- identical in all five practical sessions. Do not change.
# It rebuilds exactly the same train / dev / test split every time it is run.
# ==========================================================================
import os
import tarfile
import urllib.request
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

if not os.path.exists('reviews_train.csv'):
    url = 'http://www.ccl.kuleuven.be/Courses/compling/data/review_dataset.tar.gz'
    urllib.request.urlretrieve(url, 'review_dataset.tar.gz')
    with tarfile.open('review_dataset.tar.gz') as tar:
        tar.extractall(filter='data')

SEED = 2627  # fixed seed: everybody gets the same split, in every session

train_full = pd.read_csv('reviews_train.csv', header=0, delimiter='\t', quoting=3)
test = pd.read_csv('reviews_test.csv', header=0, delimiter='\t', quoting=3)

# development set: 2000 reviews set aside from the training data
train, dev = train_test_split(train_full, test_size=2000, random_state=SEED,
                              stratify=train_full['sentiment'])
train = train.reset_index(drop=True)
dev = dev.reset_index(drop=True)

# small fixed samples, for the models that are slow at prediction time
dev_small = dev.sample(n=200, random_state=SEED).reset_index(drop=True)
test_small = test.sample(n=500, random_state=SEED).reset_index(drop=True)

print('train:', len(train), '| dev:', len(dev), '| test:', len(test),
      '| dev_small:', len(dev_small), '| test_small:', len(test_small))


def evaluate(name, gold, predicted):
    """Print (and return) accuracy and macro-averaged F1 for a list of predictions."""
    acc = accuracy_score(gold, predicted)
    f1 = f1_score(gold, predicted, labels=[0, 1], average='macro', zero_division=0)
    print('{:45s} accuracy = {:.3f} | macro-F1 = {:.3f}'.format(name, acc, f1))
    return acc, f1


Next, we'll import the Word2Vec class from the `gensim` library using the command below. We'll also import the `logging` module, which will provide us with information during training, and the `gzip` module, which allows us to read compressed files.

In [ ]:
import logging
import gzip

from gensim.models import Word2Vec

The following command will download a sample from the English Wikipedia, consisting of 1 million lines (about 26 million words). Note that our corpus has already been tokenized (i.e. punctuation marks have been separated from the words).

In [ ]:
import os
import urllib.request

if not os.path.exists('wiki_sample_en.txt.gz'):
    urllib.request.urlretrieve('http://ccl.kuleuven.be/Courses/compling/data/wiki_sample_en.txt.gz',
                               'wiki_sample_en.txt.gz')

We can inspect the beginning of our corpus using the command below:

In [ ]:
!zcat wiki_sample_en.txt.gz|head -n20

The next command initializes the logging module, which will provide information about our training procedure.

In [ ]:
#remove any default config
for handler in logging.root.handlers[:]:
   logging.root.removeHandler(handler)

#initialize logging
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s',
                    level=logging.INFO)

Next, we define a corpus class that will allow us to sequentially iterate over the sentences in our corpus.

In [ ]:
class PlainTextCorpus(object):
    def __init__(self, fileName):
        self.fileName = fileName

    def __iter__(self):
        for line in gzip.open(self.fileName, 'rt'):
            line = [w.lower() for w in line.split()]
            yield line

And we instantiate that class using the location of our corpus.

In [ ]:
sentences = PlainTextCorpus('wiki_sample_en.txt.gz')

We're all set to start our training procedure, using the command below. We'll go through our corpus, and induce embeddings with a vector size of 100 (`gensim`'s default). In order to have reasonable estimates, we'll only take into account words that appear with a frequency of at least 250 in our corpus. We'll sequentially go through our corpus 5 times (in neural network lingo, one traversal is called an *epoch*). Note that our corpus is fairly small; we'd typically use data that is one to two orders of magnitude larger than the toy data we're using here. Still, the corpus will allow us to come up with reasonable word embedding representations.

The following command will take a couple of minutes to finish, so sit back and enjoy watching the training procedure.

In [ ]:
model = Word2Vec(sentences, min_count=250)

Once our training procedure is finished, we'll have a model that contains embeddings for a vocabulary of 8120 words, as can be confirmed with the command below.

In [ ]:
len(model.wv.index_to_key)

It is now straightforward to compute the most similar words according to our embedding model. Use the command below to compute the most similar words to the word *piano*.

In [ ]:
model.wv.most_similar('piano')

During the lecture, we saw that you can do analogy computations by carrying out vector arithmetic. Let's try that out. Note that the `Word2Vec` module encodes the computations slightly differently. For a computation like *brussels - belgium + france*, you need to provide as arguments a list of the positive terms (*brussels* and *france*), and a list of the negative terms (*belgium*).

In [ ]:
model.wv.most_similar(positive=['brussels', 'france'], negative=['belgium'])

Let's also try out the ubiquitous example:

In [ ]:
model.wv.most_similar(positive=['king', 'woman'], negative=['man'])

Finally, we'll explore a visualization of the embeddings we created. To do so, we'll make use of the `TensorBoard Projector`, the visualization module that comes with Google's `Tensorflow` library for neural network modeling. First we'll import the necessary modules (including the `os` module, which allows us to perform some system commands).

In [ ]:
import os
import tensorflow as tf
from tensorboard.plugins import projector

%load_ext tensorboard

We can now convert our embeddings into the right format, so that they can be used by `TensorBoard`. If you're interested in how that works, you can inspect the comments provided in the code below.

In [ ]:
# set up a logs directory for Tensorboard
log_dir='/logs/embeddings/'
if not os.path.exists(log_dir):
    os.makedirs(log_dir)

# save vocabulary line by line
with open(os.path.join(log_dir, 'metadata.tsv'), "w") as f:
  for w in model.wv.index_to_key:
    f.write("{}\n".format(w))

# save the embedding vectors in a variable
weights = tf.Variable(model.wv.get_normed_vectors())

# create a checkpoint for the embeddings
checkpoint = tf.train.Checkpoint(embedding=weights)
checkpoint.save(os.path.join(log_dir, "embedding.ckpt"))

# Some more configuration setup
config = projector.ProjectorConfig()
embedding = config.embeddings.add()
embedding.tensor_name = "embedding/.ATTRIBUTES/VARIABLE_VALUE"
embedding.metadata_path = 'metadata.tsv'
projector.visualize_embeddings(log_dir, config)

We're all set. We can now load `TensorBoard`, and use the `Projector` to explore the embeddings we created.

Note: if you get the error `No dashboards are active for the current dataset`, this means the interface has not yet properly loaded. Just press the play button for the same command again in order to properly load it.

In [ ]:
# Run tensorboard with the log data
%tensorboard --logdir /logs/embeddings/

The embeddings you have trained will now be displayed in a 3D visualization in the `TensorBoard Projector`. Using the panel on the right, you can search for words to find the closest neighbors to a particular word. For example, try searching for *basketball*. You are likely to see similar neighbors such as *soccer* and *football*. Note that you can use your mouse to move through the 3D space, and click on the circles to visualize the neighbors of a particular word in space.

The visualization you're seeing is a projection of your original embedding space (of 100 dimensions) to three dimensions, using a dimensionality reduction technique called Principal Component Analysis (PCA). The technique projects the original space down to three dimensions, preserving as much of the variance in the original space as possible.

### Exercise 1

Try searching for ambiguous words, such as *record* or *sentence*. What does the neighbouring space look like? Can you see clusters of related senses? You can click the `Isolate .. points` button in the top right in order to visualize only the nearest neighbours, in order to gain a clearer view of the neighbouring space. Also try out a different dimensionality reduction algorithm, such as `UMAP` (this is also a dimensionality reduction technique, but works a bit different: it tries to preserve the nearest neighbours within the original space as closely as possible; you could also try `t-SNE` - which uses a similar idea - but this technique is quite a bit slower, so be prepared to wait).

In [ ]:
# For the different kind of visualizations, you will notice that similar words
# related to different senses end up in different locations in the word space.
# For `record`, for example, words related to the music sense cluster together,
# and words related to the sports sense cluster together.

### Exercise 2

By default, the `Word2Vec` class within the `gensim` library uses *continuous bag of words* (cbow) to induce the embeddings. Change the arguments, so that your embeddings are constructed using the *skip gram* method. You can find more information about the `Word2Vec` class (including the possible arguments) in `gensim`'s [API reference](https://radimrehurek.com/gensim/models/word2vec.html).

In [ ]:
### YOUR CODE HERE

model_sg = Word2Vec(sentences, min_count=250, sg=1)

## Word embeddings as features: row 4 of our results table

In practical session 1, we started an experiment that runs through all five sessions: the same sentiment classification task, on the same data, approached with a different type of model each time. So far our results table contains a majority baseline, a hand-written sentiment lexicon (symbolic), and a logistic regression on word counts (statistical). Today we add the first **neural** ingredient: the word embeddings we have just trained.

For the bag-of-words classifier of session 1, *superb* and *excellent* are two unrelated features. If *superb* occurs in the training reviews and *stupendous* does not, the model knows nothing about *stupendous*. Word embeddings can help here: similar words have similar vectors, so what the classifier learns about one word carries over to its neighbours.

The simplest way to turn word embeddings into a representation of a whole review is the one discussed in the lecture (CBOW for sentence encoding): the **average** of the embeddings of all the words in the review.

In [ ]:
import numpy as np

def review_vector(review, wv):
    """Represent a review as the average of the embeddings of its words."""
    # our reviews have already been tokenized and lowercased;
    # words that are not in the embedding vocabulary are simply skipped
    vectors = [wv[w] for w in review.split() if w in wv]
    if len(vectors) == 0:
        return np.zeros(wv.vector_size)
    return np.mean(vectors, axis=0)


def reviews_to_matrix(reviews, wv):
    return np.array([review_vector(review, wv) for review in reviews])


train_vectors = reviews_to_matrix(train['review'], model.wv)
dev_vectors = reviews_to_matrix(dev['review'], model.wv)
print(train_vectors.shape)

Every review is now represented by a single vector of 100 numbers (instead of 20,000 word counts). We train the same classifier as in session 1, a logistic regression, on these vectors, and evaluate it on the **development set** (the test set is used once, at the end).

In [ ]:
from sklearn.linear_model import LogisticRegression

embedding_classifier = LogisticRegression(max_iter=1000)
embedding_classifier.fit(train_vectors, train['sentiment'])

evaluate('averaged wiki embeddings (cbow) + LR (dev)', dev['sentiment'],
         embedding_classifier.predict(dev_vectors));

### Exercise 3

1. Compare the score to the scores on the development set we obtained in session 1 (lexicon: about 0.70; bag of words + logistic regression: about 0.88). Is this what you expected? What could explain the difference?
2. Try to improve the result, using the development set. Some ideas: use the skip-gram embeddings of exercise 2; or use embeddings that have been trained on much more data. The cell below downloads a set of pre-trained GloVe embeddings (Pennington et al., 2014; 50 dimensions, trained on 6 billion words of Wikipedia and newswire text), which you can use in exactly the same way as `model.wv`.
3. Choose your best setup (according to the development set), and evaluate it **once** on the test set: this is row 4 of our table.
4. In session 1, we could *read* the logistic regression: each weight belonged to a word. Look at `embedding_classifier.coef_`. What can you still say about what this model has learned?

In [ ]:
import gensim.downloader

glove = gensim.downloader.load('glove-wiki-gigaword-50')
glove.most_similar('piano')

In [ ]:
### YOUR CODE HERE

# 2. we compare three sets of embeddings on the development set

embeddings = {'wiki cbow (100 dim.)': model.wv,
              'wiki skip-gram (100 dim.)': model_sg.wv,
              'glove 6B (50 dim.)': glove}

classifiers = {}
for name, wv in embeddings.items():
    clf = LogisticRegression(max_iter=1000)
    clf.fit(reviews_to_matrix(train['review'], wv), train['sentiment'])
    classifiers[name] = clf
    evaluate(name + ' (dev)', dev['sentiment'],
             clf.predict(reviews_to_matrix(dev['review'], wv)))

# 3. the pre-trained GloVe embeddings work best on the development set (about
#    0.77, against 0.73-0.74 for our own embeddings), so that is the model we
#    evaluate -- once -- on the test set

best = 'glove 6B (50 dim.)'
evaluate('4. averaged embeddings + LR (test)', test['sentiment'],
         classifiers[best].predict(reviews_to_matrix(test['review'], embeddings[best])));

#    Note that the test score (about 0.74) is lower than the development score
#    of the same model. That is no accident: we picked the best of three models
#    *because* of its development score, so that score is an optimistic
#    estimate. This is exactly why the test set exists -- and why we do not go
#    back now to try a fourth and a fifth variant on it.

# 1. All variants are somewhat better than our lexicon, but clearly worse than
#    the bag of words model of session 1. Dense, 'meaningful' vectors do not
#    automatically beat simple counts:
#    - by averaging, a review of 250 words is squeezed into 50 or 100 numbers;
#      the few words that carry the sentiment are drowned out by all the others
#      (the bag of words model has a separate weight for each of 20000 words
#      and word pairs);
#    - general-purpose embeddings encode *topical/distributional* similarity:
#      'good' and 'bad' occur in the same contexts, so their vectors are very
#      similar (try glove.similarity('good', 'bad')) -- not what you want for
#      sentiment;
#    - like bag of words, the average ignores word order ('not good').
#    Embeddings trained on much more data help a little, but they do not close
#    the gap: what matters is what is encoded, not only how much data was used.
#    The real gain of the neural paradigm comes when the embeddings are
#    contextualized and the whole network is tuned for the task -- session 3.

print(glove.similarity('good', 'bad'), glove.similarity('good', 'piano'))

# 4. The classifier still has one weight per feature, but a feature is now an
#    anonymous embedding dimension: 'dimension 37 has weight -2.1' does not tell
#    us anything. We gained generalization over similar words, but lost the
#    direct readability we had in session 1.

print(classifiers[best].coef_[0][:10])

## Learning the embeddings end-to-end: a CBOW classifier (row 5)

In row 4, the embeddings were trained beforehand (on Wikipedia, or on 6 billion words for GloVe), and then used as they were. A neural network can also **learn the embeddings itself**, as part of the classifier. The embedding layer is then simply the first layer of the network, and its values are adjusted during training, just like all the other parameters, on the training reviews. The embeddings are then no longer general-purpose word vectors, but vectors that are useful for *this* task.

We build the network with `keras`, a library for neural networks that is part of `tensorflow` (already installed on Colab). The first model is the continuous bag of words (CBOW) classifier from the lecture: an embedding layer, the average of the embeddings of all words in the review, and an output layer.

A neural network needs numbers as input, so we first turn each review into a list of word indices. This takes three small steps.

**Step 1: the vocabulary.** We number the 20,000 most frequent words of the training reviews. Index 0 is reserved for padding (see step 3), and index 1 for words that are not in the vocabulary.

In [ ]:
from collections import Counter
import numpy as np
import keras

max_features = 20000

# count how often each word occurs in the training reviews
counts = Counter()
for review in train['review']:
    counts.update(review.split())

# the vocabulary: index 0 for padding, 1 for unknown words, then the most frequent words
vocabulary = ['<pad>', '<unk>'] + [word for word, count in counts.most_common(max_features - 2)]
word_index = {word: i for i, word in enumerate(vocabulary)}

print(len(vocabulary), 'words in the vocabulary')
print(vocabulary[:12])

**Step 2: from words to indices.** The function below replaces every word of a review by its index. Words that are not in the vocabulary get index 1.

In [ ]:
def to_indices(review):
    return [word_index.get(word, 1) for word in review.split()]

print(train['review'][0][:80])
print(to_indices(train['review'][0])[:15])

**Step 3: padding.** Reviews have different lengths, but the network needs inputs of the same size. We keep the first 200 words of every review, and fill shorter reviews up with zeros at the end. This is called *padding*, and `pad_sequences` does it for us. The result is a matrix with one row per review and 200 columns.

In [ ]:
maxlen = 200

x_train = keras.utils.pad_sequences([to_indices(r) for r in train['review']], maxlen=maxlen, padding='post', truncating='post')
x_dev = keras.utils.pad_sequences([to_indices(r) for r in dev['review']], maxlen=maxlen, padding='post', truncating='post')
x_test = keras.utils.pad_sequences([to_indices(r) for r in test['review']], maxlen=maxlen, padding='post', truncating='post')

y_train = train['sentiment']
y_dev = dev['sentiment']
y_test = test['sentiment']

print('shape of the training matrix:', x_train.shape)
print(x_train[0])

Now we can build the CBOW model. We stack three layers with `keras.Sequential` (the name has nothing to do with word sequences; it just means that the layers come one after the other):

* an **embedding layer**, which turns every word index into a vector of 100 values. These vectors are the embeddings, and they are learned during training. `mask_zero=True` tells the network to ignore the padding;
* a **pooling layer**, which takes the average of the embeddings of all words in the review. The result is one vector of 100 values per review;
* an **output layer** with a single neuron and a sigmoid function, which turns the average vector into a number between 0 (negative) and 1 (positive).

In [ ]:
embedding_dims = 100

cbow = keras.Sequential([
    keras.layers.Input(shape=(maxlen,)),
    keras.layers.Embedding(len(vocabulary), embedding_dims, mask_zero=True, name='embedding'),  # word index -> vector
    keras.layers.GlobalAveragePooling1D(),                                                        # average of the vectors
    keras.layers.Dense(1, activation='sigmoid')                                                   # one output between 0 and 1
])

cbow.summary()

Before training, we *compile* the model: we choose the loss function (binary cross-entropy, which compares the output between 0 and 1 with the correct label) and the optimization method (`adam`, a variant of gradient descent). We then train for 5 *epochs* (5 passes over the training set), in batches of 32 reviews at a time. After every epoch, the accuracy on the development set is printed, so we can see whether the model is still improving.

In [ ]:
cbow.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

cbow.fit(x_train, y_train, validation_data=(x_dev, y_dev), epochs=5, batch_size=32)

The network outputs a number between 0 and 1 for every review. To put the model in our results table, we turn this into a label: 1 (positive) when the output is above 0.5, and 0 (negative) otherwise.

In [ ]:
def predict_labels(model, x):
    probabilities = model.predict(x, verbose=0)
    return (probabilities > 0.5).astype(int).flatten()

evaluate('cbow, embeddings learned from scratch (dev)', y_dev, predict_labels(cbow, x_dev));

**Second variant: start from word2vec.** Instead of random starting values, we can fill the embedding layer with the word2vec vectors we trained on Wikipedia at the start of this session (for the words that occur in both vocabularies), and let the network adapt them to the task during training. We first build a matrix with one row per vocabulary word.

In [ ]:
# a matrix with one row per vocabulary word, filled with the word2vec vectors;
# words without a word2vec vector get small random values
pretrained = np.random.normal(0, 0.1, (len(vocabulary), model.wv.vector_size))
found = 0
for i, word in enumerate(vocabulary):
    if word in model.wv:
        pretrained[i] = model.wv[word]
        found += 1

print(found, 'of', len(vocabulary), 'vocabulary words have a word2vec vector')

The model is the same as before; the only difference is that we copy the word2vec matrix into the embedding layer before training.

In [ ]:
cbow_w2v = keras.Sequential([
    keras.layers.Input(shape=(maxlen,)),
    keras.layers.Embedding(len(vocabulary), model.wv.vector_size, mask_zero=True, name='embedding'),
    keras.layers.GlobalAveragePooling1D(),
    keras.layers.Dense(1, activation='sigmoid')
])

# copy the word2vec vectors into the embedding layer
cbow_w2v.get_layer('embedding').set_weights([pretrained])

cbow_w2v.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
cbow_w2v.fit(x_train, y_train, validation_data=(x_dev, y_dev), epochs=5, batch_size=32)

evaluate('cbow, word2vec start (dev)', y_dev, predict_labels(cbow_w2v, x_dev));

We can look at the embedding space that the classifier has learned in the `TensorBoard Projector`, as we did for the word2vec space at the start of this session. The cell below writes the embedding layer of the first CBOW model (the one that started from scratch) to a separate log directory.

In [ ]:
def export_for_projector(keras_model, log_dir):
    """Write the embedding layer of a model to log_dir, for the TensorBoard Projector."""
    if not os.path.exists(log_dir):
        os.makedirs(log_dir)
    # the vocabulary, one word per line
    with open(os.path.join(log_dir, 'metadata.tsv'), 'w') as f:
        for word in vocabulary:
            f.write(word + '\n')
    # the embedding matrix, saved as a checkpoint
    weights = tf.Variable(keras_model.get_layer('embedding').get_weights()[0])
    checkpoint = tf.train.Checkpoint(embedding=weights)
    checkpoint.save(os.path.join(log_dir, 'embedding.ckpt'))
    # configuration for the Projector
    config = projector.ProjectorConfig()
    embedding = config.embeddings.add()
    embedding.tensor_name = 'embedding/.ATTRIBUTES/VARIABLE_VALUE'
    embedding.metadata_path = 'metadata.tsv'
    projector.visualize_embeddings(log_dir, config)


export_for_projector(cbow, '/logs/cbow/')

Start the `Projector` on this directory (run the command a second time if you get the `No dashboards are active` error):

In [ ]:
%tensorboard --logdir /logs/cbow/

Compare this with the word2vec space from the first part of the session. There, the words were spread more or less evenly throughout the space, with clusters of topically related words. Here, the space is dichotomous: the words are pulled towards one of two extremes. Search for a word with a positive polarity (*fantastic*, *wonderful*) and one with a negative polarity (*horrible*, *waste*) to confirm that the two extremes correspond to positive and negative sentiment. The classifier only needs one thing from the embeddings, the sentiment of a word, and training shapes the space accordingly. Words that carry no sentiment (*film*, *plot*) end up in between.

You can call `export_for_projector` on `cbow_w2v` (with another log directory) to see what remains of the word2vec structure after training on the reviews.

### Exercise 4

1. Compare the development scores of the two CBOW variants with each other and with row 4 (fixed embeddings + logistic regression). Does learning the embeddings for the task help? Does the word2vec start help?
2. Choose the better variant (on the development set) and evaluate it **once** on the test set: this is row 5 of our table.
3. The learned embeddings are task-specific. Use the function below to look up the nearest neighbours of a few words (*great*, *boring*, *film*, *plot*) in the embedding layer of the first CBOW model (`cbow`, trained from scratch), and compare with the neighbours in the word2vec model (`model.wv.most_similar`; not every word is in the Wikipedia vocabulary). What has changed, and why? You can also try the model that started from word2vec: is the picture as clear there?

In [ ]:
def nearest_neighbours(keras_model, word, n=8):
    # the embedding matrix: one row per vocabulary word
    E = keras_model.get_layer('embedding').get_weights()[0]
    # make every row of length 1, so that the dot product is the cosine similarity
    E = E / np.linalg.norm(E, axis=1, keepdims=True)
    # similarity of the word with every other word
    similarities = E @ E[word_index[word]]
    # most similar words first; the first one is the word itself, so we skip it
    best = np.argsort(-similarities)[1:n + 1]
    return [vocabulary[i] for i in best]

In [ ]:
### YOUR CODE HERE

# 1. In our run: from scratch 0.879, word2vec start 0.864 (dev accuracy), against
#    0.736 for the fixed Wikipedia embeddings + logistic regression (row 4).
#    Learning the embeddings on the task helps a lot. The word2vec start does
#    not help here: 18,000 reviews are enough to learn sentiment embeddings from
#    scratch, and the Wikipedia vectors encode topical similarity, which is not
#    what this task needs.

# 2. the better variant on the test set (look at the dev scores above;
#    change the name below if the other variant did better for you)
best_cbow = cbow
evaluate('5. cbow classifier (test)', y_test, predict_labels(best_cbow, x_test))
# In our run the test score was 0.840, a few points below the dev score: the
# dev set was used to choose the variant and the number of epochs, so the dev
# score is slightly optimistic. The test score is the one that goes in the table.

# 3. nearest neighbours in the learned embeddings (model trained from scratch) vs. word2vec
for word in ['great', 'boring', 'film', 'plot']:
    print(word)
    print('   cbow:    ', nearest_neighbours(cbow, word))
    if word in model.wv:
        print('   word2vec:', [w for w, s in model.wv.most_similar(word, topn=8)])
    else:
        print('   word2vec: (not in the Wikipedia vocabulary)')

# In the learned embeddings, words group by sentiment rather than by topic or
# word class: the neighbours of 'great' are other positive words, those of
# 'boring' other negative words, whatever their part of speech. The classifier
# only needs one thing from the embeddings, and that is what they end up
# encoding. Topic words such as 'film' or 'plot' carry little sentiment; their
# neighbours are less structured.
# In the model that started from word2vec, every word keeps part of its
# Wikipedia vector, so the space mixes topical and sentiment similarity and the
# neighbour lists are harder to read.

## A recurrent classifier: LSTM (row 6)

The CBOW model ignores word order: *not good, quite bad* and *not bad, quite good* get exactly the same average vector. A recurrent network reads the review word by word, and keeps a state that is updated after every word. We use an LSTM, and classify from its state after the last word.

In `keras`, this is one change to our model: the pooling layer (the average) is replaced by an `LSTM` layer of 64 units. The embedding layer is the same as before, and again starts from the word2vec vectors.

Training is slower than for the CBOW model, because the words have to be processed one after the other: count on a minute or two per epoch on Colab. (You can also switch to a GPU via *Runtime > Change runtime type*.)

In [ ]:
lstm = keras.Sequential([
    keras.layers.Input(shape=(maxlen,)),
    keras.layers.Embedding(len(vocabulary), model.wv.vector_size, mask_zero=True, name='embedding'),
    keras.layers.LSTM(64),                       # reads the words one by one; output: the state after the last word
    keras.layers.Dense(1, activation='sigmoid')
])
lstm.get_layer('embedding').set_weights([pretrained])

lstm.summary()

lstm.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
lstm.fit(x_train, y_train, validation_data=(x_dev, y_dev), epochs=3, batch_size=32)

evaluate('lstm (dev)', y_dev, predict_labels(lstm, x_dev));

### Exercise 5

1. Evaluate the LSTM **once** on the test set: this is row 6 of our table. How does it compare with the CBOW classifier, and with the bag-of-words model of session 1?
2. Does word order matter? Write a few short reviews in which the order is decisive (*not good, quite bad* versus *not bad, quite good*; a negative review that ends with a positive remark, ...). Turn them into a padded matrix in the same way as the reviews above (`to_indices` and `pad_sequences`), and compare the predictions of the CBOW model and the LSTM.
3. Compare the number of parameters of the two models (`count_params()`) and their training time. Is the gain, if any, worth the cost on this task?

In [ ]:
### YOUR CODE HERE

# 1. test score
evaluate('6. lstm classifier (test)', y_test, predict_labels(lstm, x_test))

# 2. word order
examples = ['not good , quite bad',
            'not bad , quite good',
            'the plot is a mess and the acting is terrible , but i loved every minute of it',
            'i loved every minute of it , but the plot is a mess and the acting is terrible']
x_examples = keras.utils.pad_sequences([to_indices(r) for r in examples], maxlen=maxlen, padding='post', truncating='post')

for text, p_cbow, p_lstm in zip(examples, predict_labels(best_cbow, x_examples), predict_labels(lstm, x_examples)):
    print('cbow:', p_cbow, '  lstm:', p_lstm, '  ', text)

# The two examples of each pair get the same prediction from the CBOW model, by
# construction: an average does not depend on the order. Whether the LSTM gets
# them right depends on the run; with three epochs on 18,000 reviews it does not
# always learn negation reliably. Check, do not assume.

# 3. parameters
print('cbow:', best_cbow.count_params(), 'parameters')
print('lstm:', lstm.count_params(), 'parameters')
# Both models have the same embedding layer (20,000 x 100 = 2 million values),
# which makes up almost all parameters; the LSTM adds about 42,000 for its gates.

## The results table

Reference numbers (your own numbers may differ slightly: the training of word embeddings is not deterministic).

| # | model | paradigm | accuracy | macro-F1 | session |
|---|---|---|---|---|---|
| 1 | majority baseline | – | 0.500 | 0.333 | 1 |
| 2 | sentiment lexicon (rules) | symbolic | 0.717 | 0.700 | 1 |
| 3 | bag of words + naive Bayes / logistic regression | statistical | 0.875 | 0.875 | 1 |
| 4 | averaged word embeddings + logistic regression | neural (static embeddings) | 0.736 | 0.736 | 2 |
| 5 | CBOW classifier, embeddings learned end-to-end | neural (learned embeddings) | 0.840 | 0.840 | 2 |
| 6 | LSTM classifier | neural (recurrent) | … | … | 2 |
| 7 | finetuned DistilBERT | neural (pretrained transformer) | *(later)* | *(later)* | 3 |
| 8 | prompted LLM (zero-/few-shot), on `test_small` | in-context | *(later)* | *(later)* | 4 |


### References

Andrew L. Maas, Raymond E. Daly, Peter T. Pham, Dan Huang, Andrew Y. Ng, and Christopher Potts. 2011. Learning Word Vectors for Sentiment Analysis. The 49th Annual Meeting of the Association for Computational Linguistics (ACL 2011).

Jeffrey Pennington, Richard Socher, and Christopher D. Manning. 2014. GloVe: Global Vectors for Word Representation. In *Proceedings of the 2014 Conference on Empirical Methods in Natural Language Processing* (EMNLP 2014).